In [5]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import random

In [2]:
spark = SparkSession.builder \
    .appName("salting") \
    .master("local[*]") \
    .config("spark.executor.memory", "512M") \
    .config("spark.executor.cores",4) \
    .config("spark.sql.adaptive.enabled", True) \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 20:08:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
spark.sparkContext.uiWebUrl

'http://macbookpro:4040'

#### Salting

* When data is skewed, spillage will happen. (data spill to disk instead of staying in memory only).

* 🧂 Salting is a technique to distribute skewed data more evenly across your processing resources. 

* When the task is complete, you can remove the added salt numbers and get your original results.

In [11]:
emp_skewed = (
    spark.range(1_000_000)
    .withColumn(
        "department_id",
        F.when(F.col("id") < 900_000,1)
        .otherwise((F.col("id") % 9 + 2).cast("int"))
)
.withColumn("first_name", F.lit("John"))
    .withColumn("last_name", F.lit("Doe"))
    .withColumn("job_title", F.lit("Engineer"))
    .withColumn("salary", F.lit(100000.0))
    .drop("id")
)

In [4]:
dept_schema = """

department_id int,
department_name string,
description string,
city string,
state string,
country string
"""

dept = spark.read.format("csv").schema(dept_schema).option("header",True).load("/Users/AnhHuynh/Documents/PySpark/department_data.txt")

In [16]:
df_joined = emp_skewed.join(dept,on=emp_skewed.department_id == dept.department_id, how="left")

df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [department_id#37], [department_id#8], LeftOuter, BuildRight, false
   :- Project [CASE WHEN (id#36L < 900000) THEN 1 ELSE cast(((id#36L % 9) + 2) as int) END AS department_id#37, John AS first_name#38, Doe AS last_name#39, Engineer AS job_title#40, 100000.0 AS salary#41]
   :  +- Range (0, 1000000, step=1, splits=8)
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=140]
      +- Filter isnotnull(department_id#8)
         +- FileScan csv [department_id#8,department_name#9,description#10,city#11,state#12,country#13] Batched: false, DataFilters: [isnotnull(department_id#8)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/AnhHuynh/Documents/PySpark/department_data.txt], PartitionFilters: [], PushedFilters: [IsNotNull(department_id)], ReadSchema: struct<department_id:int,department_name:string,description:string,city:string,state:string,c

In [15]:
emp_skewed.groupBy("department_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

+-------------+------+
|department_id| count|
+-------------+------+
|            1|900000|
|            2| 11112|
|            6| 11111|
|            3| 11111|
|            5| 11111|
|            9| 11111|
|            4| 11111|
|            8| 11111|
|            7| 11111|
|           10| 11111|
+-------------+------+



🔴 Note:

* In reality, we should enable AQE (Adaptive Query Execution)

* AQE waits until shuffle stages complete and then uses actual runtime statistics:

    * Actual partition sizes
    * Actual row counts
    * Actual data distribution
    * Skewed partitions

Then it can rewrite the plan.

